In [1]:
import functools
from pathlib import Path

import ase.io
import numpy as np
from ase import Atom
from atomse.alter import del_atoms
from atomse.layers import (
    Layers,
    O3_Stacker,
    get_stacking_vectors,
    get_stacking_vectors_from_elmt,
)
from mathull import StructureEnumerator, StructureFilter, get_species

In [2]:
def stack_layers(
    atoms, only_elmts, before_layers_cnt, stacking_vectors=None, mode="farthest"
):
    layers = Layers(atoms, only_elmts=only_elmts).layers
    if len(layers) == before_layers_cnt:
        stker = O3_Stacker(atoms, layers=layers)
        if stacking_vectors is None:
            _atoms = stker.stack_layer(0, range(len(layers))[1:], mode=mode)
        else:
            _atoms = stker.stack_layer_by_vectors(
                0, range(len(layers))[1:], vectors=stacking_vectors
            )
    else:
        idxs = []
        for idxs_in_layer in layers:
            idxs.extend(idxs_in_layer)
        _atoms = del_atoms(atoms, idxs)
    return _atoms


def get_operator(primitive_structure, elmt, layers, v_elmt):
    # 结构操作
    # structure_operator = None
    #structure_operator = functools.partial(
        #stack_layers,
        #only_elmts=[
            # "Ni",
            # "Co",
            # "Mn",
            #elmt,
            # 'H',
            # 'Ca',
        #],
        # ignore_elmts=['Li', 'O'],
        #before_layers_cnt=len(layers),
        # stacking_vectors=get_stacking_vectors(primitive_structure, 16, [23, ]),
        # stacking_vectors=get_stacking_vectors_from_elmt(primitive_structure, v_elmt),
        #mode="closest",
    #)
    #return structure_operator
    return None

def P2_stack(atoms):
    _atoms = atoms.copy()
    layers = Layers(_atoms, only_elmts=['Li', 'H']).layers
    if len(layers) < 2:
        _atoms = del_atoms(_atoms, 'Li')
    else:
        radian = np.radians(180)
        roate_matrix = np.array(
            [
                [np.cos(radian), -np.sin(radian), 0],
                [np.sin(radian), np.cos(radian), 0],
                [0, 0, 1],
            ]
        )
        for idx in layers[0]:
            atom = _atoms[idx]
            # nx = atom.position[1]
            # ny = atom.position[0]-atom.position[1]
            # nz = atom.position[2]+_atoms.cell.cartesian_positions([0,0,0.5])[2]
            # n_pos = [nx,ny,nz]
            n_pos = np.dot(roate_matrix, atom.position)
            n_pos += _atoms.cell.cartesian_positions([0, 0, 0.5])
            _atoms.append(Atom(atom.symbol, n_pos))
        del_idxs = []
        for idxs in layers[1:]:
            del_idxs.extend(idxs)
        _atoms = del_atoms(_atoms, del_idxs)
    return _atoms


# def get_operator(*args, **kwargs):
#     return P2_stack

In [3]:
def get_filter_combinations():
    f_c = (
        (
            "Ewald",
            {
                "num_outs": 5,
                "certain_charges": {
                    "Li": 1,
                    "Na": 1,
                    "K": 1,
                    "O": -2,
                    "X": 0,
                    "Bi": 3,
                    "P": 5,
                    "Cl": -1,
                    "S": -2
                },
                "enhance_ewald_repulsion": {"Li": 2, "Na": 2, "K": 3, "Ca": 2},
                "charge_rule": (
                    {
                        "elmt": "Fe",
                        "ox_change": (2, 3),
                        "field": None,
                        "ignore_fields": ["T:4"],
                    },
                    {"elmt": "Fe", "ox_change": (2, 3), "field": "T:4"},
                    {
                        "elmt": "Mn",
                        "ox_change": (2, 3),
                        "field": None,
                        "ignore_fields": ["T:4"],
                    },
                    {"elmt": "Mn", "ox_change": (2, 3), "field": "T:4"},
                    {"elmt": "Ti", "ox_change": (3, 4), "field": None},
                    {"elmt": "Mn", "ox_change": (3, 4), "field": None},
                    {"elmt": "V", "ox_change": (2, 3), "field": None},
                    {"elmt": "V", "ox_change": (3, 4), "field": None},
                    {"elmt": "V", "ox_change": (4, 5), "field": None},
                    {"elmt": "Cu", "ox_change": (2, 3), "field": None},
                    {"elmt": "Ni", "ox_change": (2, 3), "field": None},
                    {"elmt": "Ni", "ox_change": (3, 4), "field": None},
                    {"elmt": "Cr", "ox_change": (3, 4), "field": None},
                    {"elmt": "Cr", "ox_change": (4, 5), "field": None},
                    {"elmt": "Co", "ox_change": (3, 4), "field": None},
                    {
                        "elmt": "Fe",
                        "ox_change": (3, 4),
                        "field": None,
                        "ignore_fields": ["T:4"],
                    },
                    {
                        "elmt": "Fe",
                        "ox_change": (4, 5),
                        "field": None,
                        "ignore_fields": ["T:4"],
                    },
                    {"elmt": "O", "ox_change": (-2, -1), "field": None},
                    {"elmt": "Fe", "ox_change": (3, 4), "field": "T:4"},
                ),
                "fields": {
                    "Li": "O:6",
                    "Ni": "O:6",
                    "Co": "O:6",
                    "Fe": "O:6",
                    "Mn": "O:6",
                    "Zn": "O:6",
                    "O": "O:6",
                    "Na": "O:6",
                    "Ti": "O:6",
                    "Cr": "O:6",
                    "P": "O:4",
                },
                "ncores": 8,
            },
        ),
        # ("M3GNet", {"num_outs": 5, "relax": False}),
        # ("CHGNet", {"num_outs": 5, "relax": False}),
    )
    return f_c

In [4]:
def get_structures(primitive_structure, v_elmt):
    # 掺杂位置
    elmt = "Li"
    replace_elmts = ["Li", "X"]
    # replace_elmts = ["Ni", "Co", "Mn"]
    ref_elmt = ("O", 3)
    # species = get_species(primitive_structure, elmt, [elmt, "X"])
    species = get_species(primitive_structure, [elmt], [replace_elmts])
    # print(species)
    # 一层中的掺杂位置
    # layers = Layers(
    #     primitive_structure,
    #     only_elmts=[
    #         elmt,
    #     ],
    # ).layers
    # partial_species = get_species(primitive_structure, [layers[0]], [replace_elmts])
    # print(partial_species)
    # /////////////////////////////////////
    se_ = StructureEnumerator(
        primitive_structure=primitive_structure,
        species=species,
        ref_elmt=ref_elmt,
        symprec=0.05,
    )
    structures = se_.enumerate(
        # sizes=range(1),
        sizes=(1,),
        # partial_species=partial_species,
        partial_species=species,
        structure_operator=None,
        # structure_operator=get_operator(),
        # structure_operator=get_operator(primitive_structure, elmt, layers, v_elmt),
        limit_latticeC=True,
        # limit_concentration={
        # "Ca": (1/9, 1.5/9),
        # "Ni": (1/9, 1.5/9),
        # "Co": (1/9, 1.5/9),
        # "Mn": (1/9, 1.5/9),
        # "Zn": (2 / 108, 2.5 / 108),
        # },
        ncores=8,
        verbose=True,
    )
    # print(structures)
    structures_dict = se_.group_by_concentration(
        structures,
        elmt_symbols=('Li','La','O'),
        # elmt_symbols=("Li", "Ni", "Co", "Mn", "O"),
        # cutoffs={2: 7, 3: 5},
        deduplication=True,
        prefer_size="smaller",
    )
    # return structures_dict
    sf_ = StructureFilter(filter_combinations=get_filter_combinations())
    filtered_structures_dict = sf_.filter_group_structures(structures_dict)
    return filtered_structures_dict

In [5]:
def write_structures(poscar, outdir, v_elmt):
    poscar = Path(poscar)
    primitive_structure = ase.io.read(poscar)
    structures_dict = get_structures(primitive_structure, v_elmt)
    outdir = Path(outdir)
    outdir.mkdir(exist_ok=True)
    for symbol_, structures in structures_dict.items():
        for cnt, structure in enumerate(structures, start=1):
            ase.io.write(
                outdir / f"{symbol_}_{cnt:03d}.vasp", structure, direct=True, sort=True
            )
    print("Done!")

In [6]:
# rootdir = Path("/home/lilz/cac/mp_database/NCM/LCO/O3")
# for name in ["O3_12_S.vasp", "O3_12_T.vasp"]:
#     poscar = rootdir / name
#     outdir = poscar.parent / poscar.stem
#     v_elmt = "Li"
#     write_structures(poscar, outdir, v_elmt)

In [ ]:
poscar = Path("LiLa2O3.vasp")
# poscar = Path('/home/lilz/cac/mp_database/NCM/331_supercells/test.vasp')
outdir = Path('test')
# outdir = poscar.parent / "same_layer_max_to_13"
v_elmt = "V"
write_structures(poscar, outdir, v_elmt)

Enumerating structures: 100
Enumerating structures: 200
Enumerating structures: 300
Enumerating structures: 400
Enumerating structures: 500
Enumerating structures: 600
Enumerating structures: 700
Enumerating structures: 800
Enumerating structures: 900
Enumerating structures: 1000
Enumerating structures: 1100
Enumerating structures: 1200
Enumerating structures: 1300
Enumerating structures: 1400
Enumerating structures: 1500
Enumerating structures: 1600
Enumerating structures: 1700
Enumerating structures: 1800
Enumerating structures: 1900
Enumerating structures: 2000
Enumerating structures: 2100
Enumerating structures: 2200
Enumerating structures: 2300
Enumerating structures: 2400
Enumerating structures: 2500
Enumerating structures: 2600
Enumerating structures: 2700
Enumerating structures: 2800
Enumerating structures: 2900
Enumerating structures: 3000
Enumerating structures: 3100
Enumerating structures: 3200
Enumerating structures: 3300
Enumerating structures: 3400
Enumerating structures: